# 🏥 TRICARE RAG Q&A 파이프라인

**사용 데이터**
| 파일 | 형식 | 설명 |
|---|---|---|
| `Costs_Fees.pdf` | PDF | TRICARE 2026 비용·수수료 전체 표 |
| `TOP_Handbook_AUG_2023_FINAL.pdf` | PDF | TRICARE Overseas Program 핸드북 |
| `mental_health_services.csv` | CSV | 정신건강 서비스별 보장 내용 |

**사용 모델 / 도구**
| 구성 요소 | 선택 |
|---|---|
| LLM | `gpt-4o-mini` |
| 임베딩 | `BAAI/bge-m3` (다국어 100개+) |
| 벡터 DB | Chroma (로컬) |
| PDF 텍스트 | PyMuPDFLoader |
| PDF 표 추출 | pdfplumber |


## 0. 패키지 설치

처음 실행 시에만 필요합니다.


In [ ]:
# !pip install langchain langchain-community langchain-chroma langchain-anthropic \
# pymupdf pdfplumber sentence-transformers chromadb python-dotenv -q


## 1. 환경 설정

`.env` 파일 예시:
```
OPENAI_API_KEY=sk-...
```


In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

api_key = os.getenv('OPENAI_API_KEY')
print(f"✅ API Key 로드: {'성공' if api_key else '실패 → .env 파일 확인 필요'}")


## 2. 파일 경로 설정

세 가지 데이터 소스를 함께 사용합니다.
파일이 노트북과 같은 디렉터리에 있으면 파일명만 입력하면 됩니다.


In [ ]:
PDF_COSTS   = 'C:\SKN-_24\제 3차 프로젝트\data\Costs_Fees.pdf'
PDF_HANDBOOK = 'C:\SKN-_24\제 3차 프로젝트\data\TOP_Handbook_AUG_2023_FINAL_092223_508 (1).pdf'
CSV_MENTAL  = 'C:\SKN-_24\제 3차 프로젝트\data\mental_health_services.csv'

for path in [PDF_COSTS, PDF_HANDBOOK, CSV_MENTAL]:
    status = '✅' if os.path.exists(path) else '❌'
    print(f'{status} {path}')


## 3. 임베딩 모델 초기화 — `BAAI/bge-m3`

### BAAI/bge-m3

- Cigna 노트북(은우님 코드) 참고 中 모델 변경
- `BAAI/bge-m3`는 100개 이상의 언어를 **동일한 벡터 공간**에 매핑하므로
한국어 질문과 영어 문서가 높은 유사도로 매칭


In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings

# GPU가 없는 환경이면 'device': 'cpu'로 변경
embedding_model = HuggingFaceEmbeddings(
    model_name='BAAI/bge-m3',
    model_kwargs={'device': 'cpu'}
)

print('✅ BAAI/bge-m3 임베딩 모델 로드 완료')


## 4. PDF 로드 — PyMuPDFLoader

### PyMuPDFLoader

- 원본 노트북에서 `PyPDFLoader`는 복잡하게 압축된 최신 PDF에서
- 가끔 파싱 에러가 발생
- `PyMuPDFLoader`는 동일한 페이지 단위 추출 방식이지만
PDF 파싱 안정성이 더 높음

- 두 PDF를 로드한 뒤 하나의 리스트로 합쳐서 청킹
- 메타데이터의 `source` 필드로 어느 파일에서 나온 청크인지 추적할 수 있음


In [ ]:
#!pip install pymupdf

In [ ]:
from langchain_community.document_loaders import PyMuPDFLoader

loader_costs= PyMuPDFLoader(PDF_COSTS)
loader_handbook= PyMuPDFLoader(PDF_HANDBOOK)

docs_costs= loader_costs.load()
docs_handbook= loader_handbook.load()

# 두 PDF를 하나의 리스트로 합침
all_pdf_docs = docs_costs + docs_handbook


- 확인


In [ ]:
print(f'1. Costs_Fees.pdf: {len(docs_costs)}페이지')
print(f'2. TOP_Handbook.pdf: {len(docs_handbook)}페이지')
print(f'=> 합계: {len(all_pdf_docs)}페이지')
print(f'\n--- 샘플 메타데이터 ---')
print(all_pdf_docs[0].metadata)
print(all_pdf_docs[3].metadata)
print(all_pdf_docs[4].metadata)


## 5. PDF 청킹 — RecursiveCharacterTextSplitter

### chunk_size=500, chunk_overlap=50

- 원본 노트북은 보험 약관 특성상 기본값(200)보다 큰 500을 사용
- TRICARE 문서도 마찬가지로 문장이 길고 맥락이 이어지는 경우 多
- (ex: Group A / Group B 분기 설명이 한 단락에 묶여있는 경우)

- `chunk_overlap=50`은 청크 경계에서 문맥이 끊기는 것을 방지
- ex. "Group A의 본인부담금은—" 뒤에 청크가 잘리면,
- 다음 청크의 앞 50자에 해당 내용이 반복되어 맥락이 유지


1. 분활 설정


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    separators=['\n\n', '\n', '. ', ' ', '']
)

pdf_chunks = text_splitter.split_documents(all_pdf_docs)


- 확인하기


In [ ]:
print(f' 분할 결과: {len(all_pdf_docs)}페이지 → {len(pdf_chunks)}개 청크')
print(f'\n--- 청크 샘플 ---')
print(pdf_chunks[0].page_content[:300])
print(f'\nMetadata: {pdf_chunks[0].metadata}')


## 6. CSV 로드 — Row 단위 Document 변환

### CSVLoader 바로 쓰지 않고 직접 변환

- `CSVLoader`를 그냥 쓰면 row 전체를 하나의 Document로 만들어 줌
- `mental_health_services.csv`처럼 `내용` 컬럼이 매우 긴 경우
- 청킹 시 서비스명(Psychotherapy 등)이 잘려나갈 수 있음.

- 직접 변환하면 두 가지를 동시에 보장
1. `page_content`에 `서비스: ...\n내용: ...` 형식으로 항상 서비스명을 포함
2. `metadata`에도 `서비스명`을 저장해서 청킹 후에도 출처 추적 가능


1. csv 읽기


In [ ]:
import pandas as pd
from langchain_core.documents import Document

df = pd.read_csv(CSV_MENTAL, encoding='utf-8-sig')


2. Document 변환
- row 하나 -> Document 하나로 변환함
- page_content: 서비스명 직접 포함시킴
    - 각 청크안에 서비스:  Psychotherapy 남게 됨
- metadata: 서비스명 따로 저장 -> 검색 결과 출처 추적 가능함


In [ ]:
csv_docs = []
for _, row in df.iterrows():
    doc = Document(
        page_content=f"서비스: {row['서비스']}\n내용: {row['내용']}",
        metadata={
            'source': 'mental_health_services.csv',
            '서비스명': row['서비스'],
            'type': 'mental_health'
        }
    )
    csv_docs.append(doc)


3. 긴 row 추가 청킹
- mental_health_services.csv: 컬럼이 row마다 길이 다름
- 세션 종류별 설명이 긴 row: 500자 넘음 -> 잘림
- text_splitter: PDF 청킹할 때 만든거 재사용함


In [ ]:
csv_chunks = text_splitter.split_documents(csv_docs)


4. 결과 확인
- 샘플 출력
    - 확인할 포인트: page_content 앞 부분(서비스: ~) 잘 포함됐는지 
    - 확인할 포인트: metadata (서비스: ~) 잘 포함됐는지 


In [ ]:
print(f'📋 CSV row 수: {len(df)}개')
print(f'🔪 청킹 후 청크 수: {len(csv_chunks)}개')
print(f'\n--- 청크 샘플 ---')
print(csv_chunks[0].page_content[:300])
print(f'\nMetadata: {csv_chunks[0].metadata}')


In [ ]:
# 전체 청크 metadata 순서대로 출력
for i, chunk in enumerate(csv_chunks):
    print(f'[{i}] {chunk.metadata}')


## 7. 전체 청크 통합 → Chroma 벡터 저장소

### PDF& CSV -> 하나의 벡터 저장소에 넣기

- 질문 유형: "비용이 얼마야?(PDF)" / "Psychotherapy는 어떻게 보장돼?(CSV) (총 2가지)
- 사용자가 항상 어느 문서에서 답이 나올지 알 수는 없음
- 하나의 저장소로 통합하면 Retriever가 가장 관련 있는 청크를
소스 구분 없이 한 번에 찾게 설정

- 각 청크의 `metadata.source`와 `metadata.type`으로
나중에 어느 파일에서 왔는지 항상 추적할 수 있음


1. PDF 청크 + CSV 청크 합침


In [ ]:
from langchain_chroma import Chroma

# PDF 청크 + CSV 청크 합치기
all_chunks = pdf_chunks + csv_chunks

print(f'📦 총 청크 수: {len(all_chunks)}개 (PDF: {len(pdf_chunks)} + CSV: {len(csv_chunks)})')
print('⏳ 벡터 저장소 생성')


2. 백터 저장소 생성
- pdf_chunks(PDF 텍스트 청크) + csv_chunk =all_chunks
- collection_name='tricare_rag'


In [ ]:
# 백터 저장소 생성
vector_store = Chroma.from_documents(
    documents=all_chunks,
    embedding=embedding_model,
    collection_name='tricare_rag',
    # persist_directory 오타 수정: 'chrmoa_db' -> 'chroma_db'
    # tricare_exclusion.ipynb에서 불러올 때 './chroma_db'로 참조하므로 경로 일치 필요
    persist_directory='./chroma_db'
)

print(f'✅ Chroma 벡터 저장소 생성 완료')
print(f'   저장된 벡터 수: {vector_store._collection.count()}개')


## 8. Retriever 설정

### k=6 설정

- 원본 노트북(은우님 파일과 동일)에서 k=4에서 k=6으로 올린 이유와 동일
- TRICARE 문서는 **Group A / Group B**, **TRICARE Prime / Select** 등
같은 주제가 여러 청크에 분산되어 있음
- k=4면 한쪽 그룹 정보만 검색될 수 있음
- k=6으로 넉넉하게 설정 ->  LLM이 전체 맥락을 보고 답하게 함


In [ ]:
# Retriever 설정
retriever = vector_store.as_retriever(
    search_type='similarity',
    search_kwargs={'k': 6}
)

# 동작 확인
test_result = retriever.invoke('TRICARE Prime 본인부담금')

# 검색 결과 출력
print(f'🔍 테스트 검색: {len(test_result)}개 문서 검색됨')
for i, d in enumerate(test_result):
    source = d.metadata.get('source', 'unknown')
    print(f' [{i+1}] {source}: {d.page_content[:80]}...')


## 9. LLM + RAG 체인 구성

### ChatOpenAI = LLM으로 선택

### temperature=0.1 낮게 설정

- 보험 정보는 오답이 실제 피해로 이어질 수 있기 때문에.
- 해결책: 낮은 temperature는 창의적인 답변 X + 문서에 있는 내용을 최대한 그대로 전달하도록 유도


1. llm 초기화


In [ ]:
from langchain_openai import ChatOpenAI

model = ChatOpenAI(
    model='gpt-4o-mini',
    temperature=0.1
)


2. 언어 감지 함수 및 언어 맵 정의

- 사용자 질문의 언어를 유니코드 범위로 감지
- 팀원 코드(rag_utils.py) 참고하여 동일 방식 적용
- 답변 언어를 질문 언어에 맞추기 위해 필요


In [ ]:
# 언어 감지 함수
# 유니코드 범위로 한/중/일 판별, 나머지는 영어로 처리
# 프롬프트에 answer_language를 동적으로 주입하기 위해 필요
def detect_language(text: str) -> str:
    if any('\u4e00' <= c <= '\u9fff' for c in text):
        return 'zh'
    if any('\u3040' <= c <= '\u30ff' for c in text):
        return 'ja'
    if any('\uac00' <= c <= '\ud7a3' for c in text):
        return 'ko'
    return 'en'

# 언어 코드 -> 프롬프트용 언어명 매핑
# 'other'는 LLM이 질문 언어를 그대로 따라가도록 지시
LANGUAGE_NAME_MAP = {
    'ko': 'Korean',
    'en': 'English',
    'zh': 'Chinese',
    'ja': 'Japanese',
    'es': 'Spanish',
    'fr': 'French',
    'de': 'German',
    'other': 'the same language as the user\'s question',
}

print('✅ 언어 감지 함수 및 언어 맵 정의 완료')


3. format_docs 함수 정의


In [ ]:
def format_docs(docs):
    result = []
    for doc in docs:
        source = doc.metadata.get('source', 'unknown')
        page = doc.metadata.get('page', '')
        label = f'[{source}{f", p.{page+1}" if page else ""}]'
        result.append(f'{label}{doc.page_content}')
    return '\n\n'.join(result)


4. RAG 체인 생성 함수 정의


In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# RAG 체인 생성 함수
# 질문마다 언어를 감지한 뒤 프롬프트에 answer_language를 주입
# PromptTemplate 변수로 answer_language를 넘기면 체인 구조가 복잡해지므로
# f-string으로 프롬프트 문자열에 직접 삽입 후 PromptTemplate 생성
def make_rag_chain(question: str, retriever, model):
    language_code = detect_language(question)
    answer_language = LANGUAGE_NAME_MAP.get(language_code, 'English')

    PROMPT_TEMPLATE = (
        'You are a TRICARE health benefits specialist.\n'
        'Answer questions based ONLY on the provided TRICARE documents below.\n'
        'If the answer is not in the documents, say "해당 내용은 제공된 문서에서 확인되지 않습니다." '
        'in the user\'s language.\n'
        'Always mention relevant conditions such as Group A/B, plan type, or beneficiary status.\n\n'
        # 답변 언어 규칙: 질문 언어 감지 결과를 f-string으로 주입
        # 사용자가 어떤 언어로 질문하든 같은 언어로 답변하도록 LLM에 명시
        f'IMPORTANT LANGUAGE RULE:\n'
        f'- You must answer in {answer_language}.\n'
        f'- The answer language must match the user\'s question language.\n'
        f'- Do not switch languages unless necessary for official document names or insurance terms.\n\n'
        '[참고 문서]\n{context}\n\n'
        '[질문]\n{question}\n\n'
        '[답변]\n'
    )

    rag_prompt = PromptTemplate.from_template(PROMPT_TEMPLATE)

    return (
        {'question': RunnablePassthrough(), 'context': retriever | format_docs}
        | rag_prompt
        | model
        | StrOutputParser()
    )


# 표 전용 체인도 동일한 함수로 생성 가능 (table_retriever를 인자로 전달)
print('✅ make_rag_chain 함수 정의 완료')


5. 확인
- make_rag_chain(question, retriever, model)로 호출
- 질문마다 언어를 감지해서 체인을 생성하므로 다국어 질문에 자동 대응


In [ ]:
print('✅ RAG 체인 구성 완료 (retriever -> prompt -> gpt-4o-mini -> parser)')


## 10. Demo 1 — 일반 LLM vs RAG 비교

**INPUT**
- **질문**: TRICARE Prime에서 정신건강 상담(Psychotherapy)은 어떻게 보장되나요?

**OUTPUT**
- **일반 LLM**: 학습 데이터 기반 → 세부 조건(시간 제한, 세션 종류 등) 누락 가능
- **RAG**: `mental_health_services.csv`에서 정확한 내용 검색 후 답변


In [ ]:
from langchain_core.messages import HumanMessage

question = (
    'TRICARE Prime에서 정신건강 상담(Psychotherapy)은 어떻게 보장되나요? '
    '세션 종류별(개인/가족/그룹) 시간 제한과 주의사항도 알려주세요.'
)

# 팀원 출력 형식 통일: 'QUESTION:' 헤더 + '=== ANSWER ===' 구분선 구조 적용
print('=' * 100)
print('QUESTION:', question)

print('\n❓ [Demo 1 - 일반 LLM 답변] — 문서 없이 직접 응답')
print('-' * 65)
plain_response = model.invoke([HumanMessage(content=question)])
print(plain_response.content)
print('\n⚠️  주의: 위 답변은 학습 데이터 기반이며 실제 문서와 다를 수 있습니다.')


In [ ]:
# RAG 체인 생성 (질문 언어 감지 후 답변 언어 자동 설정)
rag_chain = make_rag_chain(question, retriever, model)

print('=== ANSWER ===')
print('📚 [Demo 1 - RAG 답변] — TRICARE 문서 검색 후 gpt-4o-mini 응답')
print('-' * 65)

retrieved = retriever.invoke(question)
print('🔎 검색된 청크:')
for i, chunk in enumerate(retrieved):
    src = chunk.metadata.get('source', '')
    svc = chunk.metadata.get('서비스명', '')
    print(f'  [{i+1}] {src} {f"({svc})" if svc else ""}: {chunk.page_content[:80]}...')

print()
rag_response = rag_chain.invoke(question)
print(rag_response)

# 팀원 출력 형식 통일: RETRIEVED DOCS 섹션 추가
print('\n=== RETRIEVED DOCS ===')
for doc in retrieved:
    print(doc.metadata)


## 11. Demo 2 — 표(Table) 기반 RAG

### 표를 별도로 처리

- `Costs_Fees.pdf`의 비용 테이블은
일반 텍스트 추출 시 열 정렬이 깨짐

- `pdfplumber`는 표의 셀 좌표를 인식
- `행 | 열` 구조를 유지하므로
- ex. "TRICARE Prime, Group A, 외래 진료비 = $26"처럼
정확한 수치 검색이 가능하게 함


In [ ]:
import pdfplumber

def extract_tables_from_pdf(pdf_path):
    """
    pdfplumber로 PDF 전체에서 표를 추출합니다.
    각 셀은 ' | '로 구분되어 LLM이 행/열 관계를 파악할 수 있습니다.
    """
    table_docs = []
    with pdfplumber.open(pdf_path) as pdf:
        for page_num, page in enumerate(pdf.pages):
            tables = page.extract_tables()

            for table_idx, table in enumerate(tables):
                if not table or len(table) < 2:
                    continue
                lines = []
                for row in table:
                    cleaned = [
                        str(cell).strip().replace('\n', ' ') if cell else ''
                        for cell in row
                    ]
                    lines.append(' | '.join(cleaned))

                table_text = '\n'.join(lines)

                table_docs.append(Document(
                    page_content=f'[TRICARE 비용표 | p.{page_num + 1}]\n{table_text}',
                    metadata={
                        'source': pdf_path,
                        'page': page_num + 1,
                        'type': 'cost_table',
                        'table_idx': table_idx + 1
                    }
                ))

    return table_docs

table_docs = extract_tables_from_pdf(PDF_COSTS)
print(f'📊 총 {len(table_docs)}개 표 추출 완료')
print()
print('--- 샘플 ---')
print(table_docs[0].page_content[:400])


## 12. 표 전용 벡터 저장소 구성

### 표 -> 기존 저장소와 분리

- 비용 관련 수치 질문("TRICARE Prime Group A 외래 진료비는?")
에서는 표 데이터만 검색하는 것이 훨씬 정확
- 텍스트 청크가 섞이면 표와 관련 없는 청크가
k=6 안에 들어와서 노이즈가 될 수 있음


- table_vector_store: PDF에서 pdfplumber로 추출한 표 = table_docs
- -> tricare_cost_tables
- 비용 이야기 따로 분리


In [ ]:
# 표 전용 벡터 저장소 생성
table_vector_store = Chroma.from_documents(
    documents=table_docs,
    embedding=embedding_model,
    collection_name='tricare_cost_tables',
    persist_directory='./chroma_db2'
)

table_retriever = table_vector_store.as_retriever(
    search_type='similarity',
    search_kwargs={'k': 3}
)

print(f'✅ 표 전용 벡터 저장소: {table_vector_store._collection.count()}개 벡터')
print('✅ 표 기반 RAG 체인 구성 완료')


In [ ]:
# 다음 실행 시 불러오는 방법
#vector_store = Chroma(
#    collection_name='tricare_rag',
#    embedding_function=embedding_model,
#    persist_directory='./chroma_db'
#)


## 13. Demo 2 — 표 기반 비용 Q&A


In [ ]:
cost_questions = [
    'TRICARE Prime에서 Group A 재향군인 가족의 응급실 방문 비용은 얼마인가요?',
    'TRICARE Select Group B 재향군인의 네트워크 내 전문의 외래 진료비는 얼마인가요?',
    'TRICARE Reserve Select의 월 보험료(Member only)는 얼마인가요?'
]

for q in cost_questions:
    # 팀원 출력 형식 통일: 'QUESTION:' + '=== ANSWER ===' + '=== RETRIEVED DOCS ===' 구조
    print('=' * 100)
    print('QUESTION:', q)

    # 질문마다 언어 감지 후 체인 생성 (비용 질문은 현재 한국어지만 다국어 확장 대응)
    table_rag_chain = make_rag_chain(q, table_retriever, model)
    answer = table_rag_chain.invoke(q)

    print('=== ANSWER ===')
    print(answer)

    print('\n=== RETRIEVED DOCS ===')
    for doc in table_retriever.invoke(q):
        print(doc.metadata)


## 14. 정리 및 다음 단계

### 데이터 처리 전략 요약

| 데이터 | 처리 방식 | 핵심 이유 |
|---|---|---|
| PDF (텍스트) | PyMuPDFLoader → 청킹 | 안정적 파싱, 페이지 메타데이터 자동 포함 |
| PDF (표) | pdfplumber → Document | 행/열 구조 보존, 수치 정확도 보장 |
| CSV | 직접 row 변환 → 청킹 | 서비스명을 본문+메타데이터 양쪽에 보존 |

### Demo 결과 요약

| | 일반 LLM | RAG (텍스트) | RAG (표) |
|---|---|---|---|
| 데이터 출처 | 학습 데이터 | TRICARE PDF + CSV | Costs_Fees.pdf 표 |
| 정확도 | 낮음 | 높음 | 매우 높음 (수치) |
| 한국어 질문 | 가능 | ✅ bge-m3로 지원 | ✅ bge-m3로 지원 |
| 다국어 답변 | 가능 | ✅ 언어 감지 + 프롬프트 주입 | ✅ 언어 감지 + 프롬프트 주입 |

### 향후 개선 방향

1. **메타데이터 필터링** — Group A/B, 플랜 종류 필터를 Retriever에 추가
2. **앙상블 Retriever** — BM25(키워드) + bge-m3(의미) 혼합 검색
3. **Persistent Chroma** — `persist_directory` 설정으로 재실행 시 재임베딩 생략
4. **평가 파이프라인** — RAGAS 라이브러리로 답변 품질 정량 측정
